# Notebook 01: Warehouse Data Audit & Scope Determination (Phase 1)

**Paper:** Rodrigo Furlan de Assis, Anis Assad Neto, Elias Ribeiro da Silva, Tim Wächter, Luis Antonio de Santa-Eulalia (2026), *Agentic Architecture for Progressive Autonomous Storage Location Assignment in Warehouses*, Procedia CIRP, 146, 643–648.

### Objectives of Phase 1:
1. **Load and profile** all 9 raw warehouse CSV datasets with appropriate delimiters and encodings.
2. **Audit spatial layout** (`Storage_Location.csv`), vertical levels $z$, and coordinate integrity.
3. **Audit product catalog** (`Product.csv`), ABC classification, and sectors.
4. **Compute baseline Mixing Index (MI)** and critical rates across all vertical levels ($z=1, 2, 3, 4$ and candidate scope $z \in \{3, 4\}$).
5. **Analyze transaction logs** (`Customer_Order.csv`, `Picking_Wave.csv`) and policy snapshots.
6. **Formulate justified Scope Determination** matching paper benchmarks without fabricating synthetic entities.


In [1]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Adjust path to import src modules
sys.path.append('..')

from src.data.loaders import load_raw_csv, load_all_raw_datasets, CSV_DELIMITERS
from src.data.cleaning import clean_storage_location, clean_product, clean_random_storage, parse_slot_value
from src.data.audit import run_full_data_audit, export_audit_tables
from src.model.warehouse_state import WarehouseState

print('Environment and modules loaded successfully.')


Environment and modules loaded successfully.


## 1. Comprehensive Profiling of All 9 CSV Files

In [2]:
# Execute full audit across all 9 raw files
audit_results = run_full_data_audit(raw_dir='../data/raw')
export_audit_tables(audit_results, output_dir='../outputs/tables')

# Display Data Profile Summary
profile_df = pd.DataFrame(audit_results['file_profiles'])
display_cols = ['filename', 'rows', 'columns', 'file_size_bytes', 'delimiter', 'primary_col', 'unique_primary_col', 'total_nulls', 'duplicate_rows']
profile_df[display_cols]


,filename,rows,columns,file_size_bytes,delimiter,primary_col,unique_primary_col,total_nulls,duplicate_rows
0,Storage_Location.csv,2292,5,74664,",",originalLocation,2292,0,0
1,Random_Storage.csv,2292,19,452787,",",originalLocation,2292,0,0
2,Product.csv,208,3,3977,;,Reference,208,0,0
3,Customer_Order.csv,122370,9,9460535,;,codCustomer,588,16,2559
4,Picking_Wave.csv,215192,6,9680418,;,waveNumber,9707,0,102201
5,Class_Based_Storage.csv,2341,20,576452,;,Location,2341,0,0
6,Dedicated_Storage.csv,2340,20,571374,;,Location,2340,0,0
7,Hybrid_Storage.csv,2340,20,571422,;,Location,2340,0,0
8,Support_Points_Navigation.csv,44,2,1164,;,points_specified,44,0,0


## 2. Spatial Distribution by Vertical Level $z$ and Coordinate Integrity

In [3]:
loc_audit = audit_results['storage_locations']
print(f"Total Storage Locations: {loc_audit['total_rows']}")
print("Distribution by level z:")
for z, cnt in loc_audit['z_distribution'].items():
    pct = cnt / loc_audit['total_rows'] * 100
    print(f"  Level z={z}: {cnt} locations ({pct:.2f}%)")

print(f"\nDuplicate Coordinates Found: {loc_audit['duplicate_coordinate_count']}")
for dup in loc_audit['duplicate_coordinates']:
    print(f"  Location: {dup['originalLocation']} -> (x={dup['x']}, y={dup['y']}, z={dup['z']})")


Total Storage Locations: 2292
Distribution by level z:
  Level z=1: 846 locations (36.91%)
  Level z=2: 846 locations (36.91%)
  Level z=3: 301 locations (13.13%)
  Level z=4: 299 locations (13.05%)

Duplicate Coordinates Found: 2
  Location: K-23-30 -> (x=102.0, y=810.0, z=3.0)
  Location: K-23-40 -> (x=102.0, y=810.0, z=3.0)


## 3. Product Catalog Audit (`Product.csv`)

In [4]:
prod_audit = audit_results['product_catalog']
print(f"Total Unique Product References: {prod_audit['unique_references']}")
print("ABC Classification Distribution:")
for k, v in prod_audit['abc_distribution'].items():
    pct = v / prod_audit['total_rows'] * 100
    print(f"  Class {k}: {v} products ({pct:.1f}%)")
print(f"All sectors belong to 'PF': {prod_audit['all_sectors_pf']}")


Total Unique Product References: 208
ABC Classification Distribution:
  Class C: 94 products (45.2%)
  Class B: 70 products (33.7%)
  Class A: 44 products (21.2%)
All sectors belong to 'PF': True


## 4. Multi-Scope Baseline Mixing Index (MI) Calculation

We compute Mixing Index across all scopes:
$$MI_l(S_t) = 1 - \frac{\max_i n_{i,l,t}}{\sum_i n_{i,l,t}}$$


In [5]:
scope_df = pd.DataFrame(audit_results['scopes_and_baseline_mi'])
scope_df[['scope_name', 'locations_count', 'total_slots', 'unique_references', 'size_min', 'size_max', 'average_mi', 'critical_locations_pct']]


,scope_name,locations_count,total_slots,unique_references,size_min,size_max,average_mi,critical_locations_pct
0,"All Levels (z in {1, 2, 3, 4})",2292,41256,208,4.0,280.0,0.4362,47.69
1,Level z=1,846,15228,208,4.0,235.0,0.2692,29.43
2,Level z=2,846,15228,208,4.0,280.0,0.2646,28.96
3,Level z=3,301,5418,208,4.0,16.0,0.9131,99.67
4,Level z=4,299,5382,208,4.0,16.0,0.9142,100.00
5,"Candidate Scope (z in {3, 4})",600,10800,208,4.0,16.0,0.9136,99.83


## 5. Scope Determination Justification: Paper Target vs Real Dataset

Comparing the paper reported specifications with our candidate scopes:


In [6]:
justification_df = pd.read_csv('../outputs/tables/scope_justification.csv')
justification_df


,Scope,Locations,SKUs,Slots_Per_Loc,Total_Slots,Size_Range,Baseline_MI,Critical_Rate_Pct,Assessment
0,"Paper Target (Assis et al., 2026)",847,209,18,15246,4.0 - 16.0,0.9180,100.00,Original reported benchmark in paper
1,Full Dataset (All z in {1..4}),2292,208,18,41256,4.0 - 280.0,0.4362,47.69,Mismatched size scale (mm/US) and dedicated zo...
2,"Candidate Scope z in {3, 4}",600,208,18,10800,4.0 - 16.0,0.9136,99.83,"RECOMMENDED: Exact size match, baseline MI wit..."
3,Sensitivity Scope z=3,301,208,18,5418,4.0 - 16.0,0.9131,99.67,Sub-floor sensitivity analysis
4,Sensitivity Scope z=4,299,208,18,5382,4.0 - 16.0,0.9142,100.00,Sub-floor sensitivity analysis


## 6. Audit of Inbound Proxy (`Customer_Order.csv`) and Outbound Logs (`Picking_Wave.csv`)

In [7]:
order_audit = audit_results['customer_order']
print(f"Customer Order Total Rows: {order_audit['total_rows']:,}")
print(f"Date Range: {order_audit['date_min']} to {order_audit['date_max']}")
print(f"Rows with quantity <= 0: {order_audit['zero_or_negative_quantities']}")
print(f"Rows with quantity > 36: {order_audit['quantities_gt_36']}")
print(f"Rows with null Size (US): {order_audit['null_sizes_count']}")

pick_audit = audit_results['picking_wave']
print(f"\nPicking Wave Total Rows: {pick_audit['total_rows']:,}")
print(f"Quantity to Pick Distribution: {pick_audit['quantity_to_pick_distribution']}")
print(f"Are all picks single units (qty=1)? {pick_audit['all_picks_equal_one']}")


Customer Order Total Rows: 122,370
Date Range: 2023-01-05 07:42:00 to 2023-10-19 07:18:00
Rows with quantity <= 0: 33
Rows with quantity > 36: 1
Rows with null Size (US): 16

Picking Wave Total Rows: 215,192
Quantity to Pick Distribution: {1: 215192}
Are all picks single units (qty=1)? True


## 7. Storage Policy Snapshots Comparison

In [8]:
pol_df = pd.DataFrame(audit_results['policy_snapshots'])
pol_df[['policy_filename', 'total_locations', 'slots_per_location', 'extra_locations_vs_random', 'unique_references', 'extra_references_vs_product']]


,policy_filename,total_locations,slots_per_location,extra_locations_vs_random,unique_references,extra_references_vs_product
0,Dedicated_Storage.csv,2340,18,48,208,0
1,Hybrid_Storage.csv,2340,18,48,208,0
2,Class_Based_Storage.csv,2341,18,49,317,109


## 8. Phase 1 Completion Summary

- **Primary Simulation Universe:** Confirmed as **`z in {3, 4}`** (600 locations, 10,800 slots, 208 SKUs, sizes 4–16, baseline MI = 0.9136, critical rate = 99.83%).
- **Sensitivity Scopes:** Defined on `z=3` (301 locations) and `z=4` (299 locations).
- **Documented Discrepancies:** Fully audited (600 vs 847 locations, 208 vs 209 SKUs, 33 zero-quantity order rows, 100% single-unit picks).
- **Phase 1 Deliverables Ready:** `data_audit.md`, CSV tables in `outputs/tables/`, and test suite in `tests/test_data_validation.py`.
